<a href="https://colab.research.google.com/github/nashranoor98/hospital-readmission-prediction/blob/main/CaseStudy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hospital Readmission Prediction
Predicting 30-day hospital readmission using Logistic Regression with L2 regularization.


## 1. Importing and Studying Data


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATHS = [
    "data/diabetic_data.csv",
    "/content/data/diabetic_data.csv",
    "/content/hospital_data/diabetic_data.csv",
    "diabetic_data.csv"
]
DATA_URL = "https://raw.githubusercontent.com/moggirain/Hospital_readmission_prediction/master/diabetic_data.csv"
for path in DATA_PATHS:
    if os.path.exists(path):
        DATA_PATH = path
        break
else:
    DATA_PATH = DATA_URL

data = pd.read_csv(DATA_PATH).replace("?", np.nan)
print("Dataset shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())


Dataset shape: (101766, 50)

First 5 rows:
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)    NaN   
1        149190     55629189        Caucasian  Female  [10-20)    NaN   
2         64410     86047875  AfricanAmerican  Female  [20-30)    NaN   
3        500364     82442376        Caucasian    Male  [30-40)    NaN   
4         16680     42519267        Caucasian    Male  [40-50)    NaN   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metformin  \
0                 1  .

In [2]:
print(data.info())
print("\nDescriptive statistics:")
print(data.describe(include="all").T.head(15))
print("\nMissing values (top 10):")
print(data.isnull().sum().sort_values(ascending=False).head(10))
print("\nDuplicate rows:", data.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      99493 non-null   str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    3197 non-null    str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                61510 non-null   str  
 11  medical_specialty         51817 non-null   str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

                             count unique               top   freq  \
encounter_id              101766.0    NaN               NaN    NaN   
patient_nbr               101766.0    NaN               NaN    NaN   
race                         99493      5         Caucasian  76099   
gender                      101766      3            Female  54708   
age                         101766     10           [70-80)  26068   
weight                        3197      9          [75-100)   1336   
admission_type_id         101766.0    NaN               NaN    NaN   
discharge_disposition_id  101766.0    NaN               NaN    NaN   
admission_source_id       101766.0    NaN               NaN    NaN   
time_in_hospital          101766.0    NaN               NaN    NaN   
payer_code                   61510     17                MC  32439   
medical_specialty            51817     72  InternalMedicine  14635   
num_lab_procedures        101766.0    NaN               NaN    NaN   
num_procedures      

## 2. EDA and Visualisation
The following graphs are generated from the hospital dataset used for this case study.


### Readmission Distribution

![Readmission Distribution](graphs/01_readmission_distribution.svg)


### Age Group Distribution

![Age Group Distribution](graphs/02_age_group_distribution.svg)


### Numeric Feature Distributions

![Numeric Feature Distributions](graphs/03_numeric_distributions.svg)


### Gender Distribution

![Gender Distribution](graphs/04_gender_distribution.svg)


### Correlation Heatmap

![Correlation Heatmap](graphs/05_correlation_heatmap.svg)


In [3]:
print(data["readmitted"].value_counts())

numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient"
]


readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64


## 3. Splitting and Scaling


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data["target_30day"] = (data["readmitted"] == "<30").astype(int)
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "target_30day"]
X = data.drop(columns=drop_cols)
y = data["target_30day"]
high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Features retained:", X.shape[1])


Training samples: 81412
Testing samples: 20354
Features retained: 45


## 4. Training and Evaluating Model


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="l2", C=1.0, max_iter=1000, solver="liblinear"))
])
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


/opt/hostedtoolcache/Python/3.11.16/x64/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


ROC-AUC: 0.6462

Confusion Matrix:
[[18039    44]
 [ 2229    42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.49      0.02      0.04      2271

    accuracy                           0.89     20354
   macro avg       0.69      0.51      0.49     20354
weighted avg       0.85      0.89      0.84     20354



### ROC Curve

![ROC Curve](graphs/06_roc_curve.svg)


### Confusion Matrix

![Confusion Matrix](graphs/07_confusion_matrix.svg)


## 5. False Negative vs False Positive
False negatives can be clinically costly because a high-risk patient may be missed. False positives may lead to additional follow-up or resource use. Therefore, threshold selection should consider the clinical cost of missed readmissions.

### Verified executed results
- Dataset: **101,766 rows × 50 columns**
- 30-day target: **11,357 positive / 90,409 negative**
- Train/test split: **81,412 / 20,354**
- ROC-AUC: **0.6462**
- Confusion matrix: **[[18039, 44], [2229, 42]]**
